In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

matches = pd.read_csv("../data/raw/international_results.csv")

matches["date"] = pd.to_datetime(matches["date"])

matches = matches.sort_values("date").reset_index(drop=True)

matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
def match_result(home_score, away_score):
    if home_score > away_score:
        return "home_win"

    if home_score < away_score:
        return "away_win"

    return "draw"


matches["result"] = matches.apply(
    lambda row: match_result(row["home_score"], row["away_score"]),
    axis=1
)

matches["result"].value_counts()

result
home_win    24248
away_win    13981
draw        11267
Name: count, dtype: int64

In [3]:
def get_recent_form(matches, team, current_date, n_matches=5):
    previous_matches = matches[
        (
            (matches["home_team"] == team) |
            (matches["away_team"] == team)
        ) &
        (matches["date"] < current_date)
    ].sort_values("date")

    recent_matches = previous_matches.tail(n_matches)

    wins = 0
    draws = 0
    losses = 0
    goals_for = 0
    goals_against = 0

    for _, match in recent_matches.iterrows():

        if match["home_team"] == team:
            goals_for += match["home_score"]
            goals_against += match["away_score"]

            if match["home_score"] > match["away_score"]:
                wins += 1
            elif match["home_score"] < match["away_score"]:
                losses += 1
            else:
                draws += 1

        else:
            goals_for += match["away_score"]
            goals_against += match["home_score"]

            if match["away_score"] > match["home_score"]:
                wins += 1
            elif match["away_score"] < match["home_score"]:
                losses += 1
            else:
                draws += 1

    games_played = len(recent_matches)

    if games_played == 0:
        return {
            "win_rate": 0,
            "goals_for": 0,
            "goals_against": 0,
        }

    return {
        "win_rate": wins / games_played,
        "goals_for": goals_for / games_played,
        "goals_against": goals_against / games_played,
    }

In [5]:
from collections import defaultdict, deque

team_history = defaultdict(lambda: deque(maxlen=5))

recent_win_rate_difference = []
recent_goals_for_difference = []
recent_goals_against_difference = []

for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_history = team_history[home_team]
    away_history = team_history[away_team]

    def summarize_history(history):
        if len(history) == 0:
            return {
                "win_rate": 0,
                "goals_for": 0,
                "goals_against": 0,
            }

        wins = sum(game["result"] == "win" for game in history)
        goals_for = sum(game["goals_for"] for game in history)
        goals_against = sum(game["goals_against"] for game in history)

        return {
            "win_rate": wins / len(history),
            "goals_for": goals_for / len(history),
            "goals_against": goals_against / len(history),
        }

    home_form = summarize_history(home_history)
    away_form = summarize_history(away_history)

    recent_win_rate_difference.append(
        home_form["win_rate"] - away_form["win_rate"]
    )

    recent_goals_for_difference.append(
        home_form["goals_for"] - away_form["goals_for"]
    )

    recent_goals_against_difference.append(
        home_form["goals_against"] - away_form["goals_against"]
    )

    if match["home_score"] > match["away_score"]:
        home_result = "win"
        away_result = "loss"
    elif match["home_score"] < match["away_score"]:
        home_result = "loss"
        away_result = "win"
    else:
        home_result = "draw"
        away_result = "draw"

    team_history[home_team].append({
        "result": home_result,
        "goals_for": match["home_score"],
        "goals_against": match["away_score"],
    })

    team_history[away_team].append({
        "result": away_result,
        "goals_for": match["away_score"],
        "goals_against": match["home_score"],
    })

matches["recent_win_rate_difference"] = recent_win_rate_difference
matches["recent_goals_for_difference"] = recent_goals_for_difference
matches["recent_goals_against_difference"] = recent_goals_against_difference

matches[
    [
        "home_team",
        "away_team",
        "recent_win_rate_difference",
        "recent_goals_for_difference",
        "recent_goals_against_difference",
    ]
].head()

,home_team,away_team,recent_win_rate_difference,recent_goals_for_difference,recent_goals_against_difference
0,Scotland,England,0.0,0.000000,0.000000
1,England,Scotland,0.0,0.000000,0.000000
2,Scotland,England,-0.5,-1.000000,1.000000
3,England,Scotland,0.0,0.333333,-0.333333
4,Scotland,England,0.0,-0.250000,0.250000


In [6]:
last_match_date = {}

home_rest_days = []
away_rest_days = []
rest_days_difference = []

for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]
    match_date = match["date"]

    if home_team in last_match_date:
        home_days = (match_date - last_match_date[home_team]).days
    else:
        home_days = 30

    if away_team in last_match_date:
        away_days = (match_date - last_match_date[away_team]).days
    else:
        away_days = 30

    home_rest_days.append(home_days)
    away_rest_days.append(away_days)
    rest_days_difference.append(home_days - away_days)

    last_match_date[home_team] = match_date
    last_match_date[away_team] = match_date

matches["home_rest_days"] = home_rest_days
matches["away_rest_days"] = away_rest_days
matches["rest_days_difference"] = rest_days_difference

matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_rest_days",
        "away_rest_days",
        "rest_days_difference",
    ]
].head()

,date,home_team,away_team,home_rest_days,away_rest_days,rest_days_difference
0,1872-11-30,Scotland,England,30,30,0
1,1873-03-08,England,Scotland,98,98,0
2,1874-03-07,Scotland,England,364,364,0
3,1875-03-06,England,Scotland,364,364,0
4,1876-03-04,Scotland,England,364,364,0


In [7]:
def get_tournament_importance(tournament):
    tournament_lower = tournament.lower()

    if tournament == "FIFA World Cup":
        return 4

    if "qualification" in tournament_lower:
        return 3

    if tournament == "Friendly":
        return 1

    return 2


matches["is_world_cup"] = (
    matches["tournament"] == "FIFA World Cup"
).astype(int)

matches["is_qualification"] = (
    matches["tournament"].str.lower().str.contains("qualification")
).astype(int)

matches["is_friendly"] = (
    matches["tournament"] == "Friendly"
).astype(int)

matches["tournament_importance"] = matches["tournament"].apply(
    get_tournament_importance
)

matches[
    [
        "tournament",
        "is_world_cup",
        "is_qualification",
        "is_friendly",
        "tournament_importance",
    ]
].head()

,tournament,is_world_cup,is_qualification,is_friendly,tournament_importance
0,Friendly,0,0,1,1
1,Friendly,0,0,1,1
2,Friendly,0,0,1,1
3,Friendly,0,0,1,1
4,Friendly,0,0,1,1


In [8]:
matches["recent_goal_difference_difference"] = (
    matches["recent_goals_for_difference"]
    - matches["recent_goals_against_difference"]
)

matches[
    [
        "home_team",
        "away_team",
        "recent_goals_for_difference",
        "recent_goals_against_difference",
        "recent_goal_difference_difference",
    ]
].head()

,home_team,away_team,recent_goals_for_difference,recent_goals_against_difference,recent_goal_difference_difference
0,Scotland,England,0.000000,0.000000,0.000000
1,England,Scotland,0.000000,0.000000,0.000000
2,Scotland,England,-1.000000,1.000000,-2.000000
3,England,Scotland,0.333333,-0.333333,0.666667
4,Scotland,England,-0.250000,0.250000,-0.500000


In [9]:
from collections import defaultdict

team_streaks = defaultdict(int)

home_streak = []
away_streak = []
streak_difference = []

for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    current_home_streak = team_streaks[home_team]
    current_away_streak = team_streaks[away_team]

    home_streak.append(current_home_streak)
    away_streak.append(current_away_streak)
    streak_difference.append(current_home_streak - current_away_streak)

    if match["home_score"] > match["away_score"]:
        team_streaks[home_team] = max(current_home_streak, 0) + 1
        team_streaks[away_team] = min(current_away_streak, 0) - 1

    elif match["home_score"] < match["away_score"]:
        team_streaks[home_team] = min(current_home_streak, 0) - 1
        team_streaks[away_team] = max(current_away_streak, 0) + 1

    else:
        team_streaks[home_team] = 0
        team_streaks[away_team] = 0

matches["home_streak"] = home_streak
matches["away_streak"] = away_streak
matches["streak_difference"] = streak_difference

matches[
    [
        "home_team",
        "away_team",
        "home_streak",
        "away_streak",
        "streak_difference",
    ]
].head()

,home_team,away_team,home_streak,away_streak,streak_difference
0,Scotland,England,0,0,0
1,England,Scotland,0,0,0
2,Scotland,England,-1,1,-2
3,England,Scotland,-1,1,-2
4,Scotland,England,0,0,0


In [10]:
elo = {}

home_elo_before = []
away_elo_before = []


def get_k_factor(tournament):
    if tournament == "FIFA World Cup":
        return 40

    if "qualification" in tournament.lower():
        return 30

    if tournament == "Friendly":
        return 10

    return 20


def get_rating(team):
    if team not in elo:
        elo[team] = 1500

    return elo[team]


def expected_score(team_rating, opponent_rating):
    return 1 / (1 + 10 ** ((opponent_rating - team_rating) / 400))


def actual_score(team_goals, opponent_goals):
    if team_goals > opponent_goals:
        return 1

    if team_goals == opponent_goals:
        return 0.5

    return 0


def update_rating(old_rating, expected, actual, k):
    return old_rating + k * (actual - expected)


for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    home_elo_before.append(home_rating)
    away_elo_before.append(away_rating)

    expected_home = expected_score(home_rating, away_rating)
    expected_away = expected_score(away_rating, home_rating)

    actual_home = actual_score(
        match["home_score"],
        match["away_score"]
    )

    actual_away = actual_score(
        match["away_score"],
        match["home_score"]
    )

    k = get_k_factor(match["tournament"])

    elo[home_team] = update_rating(
        home_rating,
        expected_home,
        actual_home,
        k
    )

    elo[away_team] = update_rating(
        away_rating,
        expected_away,
        actual_away,
        k
    )


matches["home_elo_before"] = home_elo_before
matches["away_elo_before"] = away_elo_before

matches["elo_difference"] = (
    matches["home_elo_before"]
    - matches["away_elo_before"]
)

matches[
    [
        "home_team",
        "away_team",
        "home_elo_before",
        "away_elo_before",
        "elo_difference",
    ]
].head()

,home_team,away_team,home_elo_before,away_elo_before,elo_difference
0,Scotland,England,1500.000000,1500.000000,0.000000
1,England,Scotland,1500.000000,1500.000000,0.000000
2,Scotland,England,1495.000000,1505.000000,-10.000000
3,England,Scotland,1499.856128,1500.143872,-0.287744
4,Scotland,England,1500.139731,1499.860269,0.279462


In [11]:
from collections import defaultdict

head_to_head = defaultdict(lambda: {"team_1_wins": 0, "team_2_wins": 0, "draws": 0})

head_to_head_difference = []

for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    matchup_key = tuple(sorted([home_team, away_team]))

    team_1 = matchup_key[0]
    team_2 = matchup_key[1]

    matchup_record = head_to_head[matchup_key]

    if home_team == team_1:
        home_previous_wins = matchup_record["team_1_wins"]
        away_previous_wins = matchup_record["team_2_wins"]
    else:
        home_previous_wins = matchup_record["team_2_wins"]
        away_previous_wins = matchup_record["team_1_wins"]

    head_to_head_difference.append(
        home_previous_wins - away_previous_wins
    )

    if match["home_score"] > match["away_score"]:
        if home_team == team_1:
            matchup_record["team_1_wins"] += 1
        else:
            matchup_record["team_2_wins"] += 1

    elif match["home_score"] < match["away_score"]:
        if away_team == team_1:
            matchup_record["team_1_wins"] += 1
        else:
            matchup_record["team_2_wins"] += 1

    else:
        matchup_record["draws"] += 1

matches["head_to_head_difference"] = head_to_head_difference

matches[
    [
        "home_team",
        "away_team",
        "head_to_head_difference",
    ]
].head()

,home_team,away_team,head_to_head_difference
0,Scotland,England,0
1,England,Scotland,0
2,Scotland,England,-1
3,England,Scotland,0
4,Scotland,England,0


In [12]:
attack_rating = defaultdict(lambda: 1.0)
defense_rating = defaultdict(lambda: 1.0)

home_attack_before = []
away_attack_before = []
home_defense_before = []
away_defense_before = []

learning_rate = 0.05

for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_attack = attack_rating[home_team]
    away_attack = attack_rating[away_team]

    home_defense = defense_rating[home_team]
    away_defense = defense_rating[away_team]

    home_attack_before.append(home_attack)
    away_attack_before.append(away_attack)

    home_defense_before.append(home_defense)
    away_defense_before.append(away_defense)

    expected_home_goals = home_attack / away_defense
    expected_away_goals = away_attack / home_defense

    home_goal_error = match["home_score"] - expected_home_goals
    away_goal_error = match["away_score"] - expected_away_goals

    attack_rating[home_team] += learning_rate * home_goal_error
    attack_rating[away_team] += learning_rate * away_goal_error

    defense_rating[home_team] -= learning_rate * away_goal_error
    defense_rating[away_team] -= learning_rate * home_goal_error

    attack_rating[home_team] = max(0.1, attack_rating[home_team])
    attack_rating[away_team] = max(0.1, attack_rating[away_team])

    defense_rating[home_team] = max(0.1, defense_rating[home_team])
    defense_rating[away_team] = max(0.1, defense_rating[away_team])


matches["home_attack_before"] = home_attack_before
matches["away_attack_before"] = away_attack_before

matches["home_defense_before"] = home_defense_before
matches["away_defense_before"] = away_defense_before

matches["attack_rating_difference"] = (
    matches["home_attack_before"]
    - matches["away_attack_before"]
)

matches["defense_rating_difference"] = (
    matches["home_defense_before"]
    - matches["away_defense_before"]
)

matches[
    [
        "home_team",
        "away_team",
        "attack_rating_difference",
        "defense_rating_difference",
    ]
].head()

,home_team,away_team,attack_rating_difference,defense_rating_difference
0,Scotland,England,0.000000,0.000000
1,England,Scotland,0.000000,0.000000
2,Scotland,England,-0.100000,-0.100000
3,England,Scotland,0.038776,0.038776
4,Scotland,England,-0.034255,-0.034255


In [13]:
matches["home_advantage"] = (~matches["neutral"]).astype(int)

features = [
    "elo_difference",
    "home_advantage",
    "recent_win_rate_difference",
    "recent_goals_for_difference",
    "recent_goals_against_difference",
    "recent_goal_difference_difference",
    "rest_days_difference",
    "streak_difference",
    "is_world_cup",
    "is_qualification",
    "is_friendly",
    "tournament_importance",
    "head_to_head_difference",
    "attack_rating_difference",
    "defense_rating_difference",
]

model_data = matches[
    [
        "date",
        "home_team",
        "away_team",
        "tournament",
        "result",
    ] + features
].copy()

model_data.head()

,date,home_team,away_team,tournament,result,elo_difference,home_advantage,recent_win_rate_difference,recent_goals_for_difference,recent_goals_against_difference,recent_goal_difference_difference,rest_days_difference,streak_difference,is_world_cup,is_qualification,is_friendly,tournament_importance,head_to_head_difference,attack_rating_difference,defense_rating_difference
0,1872-11-30,Scotland,England,Friendly,draw,0.000000,1,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0.000000,0.000000
1,1873-03-08,England,Scotland,Friendly,home_win,0.000000,1,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0.000000,0.000000
2,1874-03-07,Scotland,England,Friendly,home_win,-10.000000,1,-0.5,-1.000000,1.000000,-2.000000,0,-2,0,0,1,1,-1,-0.100000,-0.100000
3,1875-03-06,England,Scotland,Friendly,draw,-0.287744,1,0.0,0.333333,-0.333333,0.666667,0,-2,0,0,1,1,0,0.038776,0.038776
4,1876-03-04,Scotland,England,Friendly,home_win,0.279462,1,0.0,-0.250000,0.250000,-0.500000,0,0,0,0,1,1,0,-0.034255,-0.034255


In [14]:
model_data.to_csv(
    "../data/processed/advanced_match_features.csv",
    index=False
)

matches.to_csv(
    "../data/processed/matches_with_advanced_features.csv",
    index=False
)